# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Cite as: {metadata.cite_as}")

## 2. Data Overview
Review available record sets, their fields, and corresponding IDs.

**Note:** All dataset entities are referenced by their `@id` as defined in the Croissant schema.

In [ ]:
# List all available record sets and their fields using `@id`
from pprint import pprint

record_sets = list(dataset.record_sets)
print(f"Record sets found: {len(record_sets)}\n")

record_set_ids = []
for record_set in record_sets:
    print(f"Record Set Name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {record_set.description}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()
    record_set_ids.append(record_set.id)
# Save a default record set for later use (use the first one if present)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
else:
    main_record_set_id = None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract data for each available record set using its @id
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for record set '@id': {rs_id}")
    except Exception as e:
        print(f"Failed loading record set {rs_id}: {e}")
if main_record_set_id is not None and main_record_set_id in dataframes:
    print(f"\nColumns in record set '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming distributions, or grouping data by key attributes using field `@id`s referenced earlier.

In [ ]:
import numpy as np

# Choose a numeric field for filtering and normalization from the main record set fields
# Please set these variables as needed based on the overview above!
# For demonstration, we try some common column keys:
if main_record_set_id is not None and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Try to find a numeric column by type
    # You may need to adjust 'accession', 'Coverage (%)', 'MW (kDa)', 'normalized abundance: sample A', etc. as per dataset
    sample_numeric_candidates = [col for col in df.columns if (df[col].dtype in [np.float64, np.int64, float, int] or 'abundance' in col)]
    if sample_numeric_candidates:
        numeric_field = sample_numeric_candidates[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
        # Filter rows with numeric field above mean
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (total: {len(filtered_df)})")
        display(filtered_df.head())

        # Normalize
        filtered_df[numeric_field + "_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

        # Try grouping by a categorical field (e.g., 'description', 'accession', or any non-numeric field)
        categorical_candidates = [
            col for col in df.columns if df[col].dtype == object and col != numeric_field
        ]
        if categorical_candidates:
            group_field = categorical_candidates[0]
            print(f"Grouping by '{group_field}' and calculating mean of numeric fields:")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No obvious numeric field found for EDA.")
else:
    print("No main DataFrame available for analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and main_record_set_id in dataframes and sample_numeric_candidates:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.xlabel(numeric_field)
    plt.title(f"Distribution of {numeric_field}")
    plt.show()
    # Try boxplot grouped by a categorical field, if available
    if categorical_candidates:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, inspect, and analyze a mass spectrometry protein dataset curated with the Croissant schema using the `mlcroissant` library. We explored record set and field structure (accessed by their `@id`), extracted records for analysis, performed basic normalization and aggregation, and visualized feature distributions. For deeper biological insights or machine learning tasks, further domain-specific feature engineering should be applied.